In [1]:
import pandas as pd
import numpy as np
import json

from huggingface_hub import hf_hub_download
from transformers import AutoConfig, AutoModel, AutoTokenizer

/home/eugenie-modolo/Documents/DL_multimodal/cancer_survival/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from platform import python_version
python_version()

'3.11.14'

In [2]:
model_name = "InstaDeepAI/BulkRNABert"

# Load model and tokenizer.
config = AutoConfig.from_pretrained(
    model_name,
    trust_remote_code=True,
)
config.embeddings_layers_to_save = (4,) # last transformer layer

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, cache_dir="../model")
model = AutoModel.from_pretrained(
    model_name,
    config=config,
    trust_remote_code=True,
    cache_dir="../model"
)

# Save the model and tokenizer to the 'model' folder
model.save_pretrained("../model/BulkRNABert")
tokenizer.save_pretrained("../model/BulkRNABert")


TypeError: BulkRNABertConfig.__post_init__() got an unexpected keyword argument 'attention_maps_to_save'

In [22]:
# Load model and tokenizer
# config = AutoConfig.from_pretrained("InstaDeepAI/BulkRNABert", trust_remote_code=True)
# config.embeddings_layers_to_save = (4,)  # last transformer layer

# Try loading with ignore of unknown params
from transformers.configuration_utils import PreTrainedConfig
try:
    config = AutoConfig.from_pretrained(
        "InstaDeepAI/BulkRNABert", 
        trust_remote_code=True
    )
except TypeError as e:
    if "attention_maps_to_save" in str(e):
        # Load config and manually remove the problematic param
        
        config_file = hf_hub_download(
            repo_id="InstaDeepAI/BulkRNABert",
            filename="config.json",
            repo_type="model"
        )
        
        with open(config_file) as f:
            config_dict = json.load(f)
        
        # Remove problematic parameters
        config_dict.pop("attention_maps_to_save", None)
        config_dict.pop("embeddings_layers_to_save", None)
        
        # Load config from dict
        config = PreTrainedConfig.from_dict(config_dict)
    else:
        raise

tokenizer = AutoTokenizer.from_pretrained(
    "InstaDeepAI/BulkRNABert", 
    config=config,  # Pass the cleaned config here
    trust_remote_code=True
)
model = AutoModel.from_pretrained("InstaDeepAI/BulkRNABert", config=config, trust_remote_code=True)

AttributeError: BinnedOmicTokenizer has no attribute _special_tokens_map

In [13]:
config

PreTrainedConfig {
  "architectures": [
    "BulkRNABert"
  ],
  "auto_map": {
    "AutoConfig": "bulkrnabert.BulkRNABertConfig",
    "AutoModel": "bulkrnabert.BulkRNABert"
  },
  "dtype": "float32",
  "embed_dim": 256,
  "ffn_embed_dim": 512,
  "init_gene_embed_dim": 200,
  "key_size": 32,
  "n_expressions_bins": 66,
  "n_genes": 19062,
  "num_attention_heads": 8,
  "num_layers": 4,
  "project_gene_embedding": true,
  "transformers_version": "5.10.0.dev0",
  "use_gene_embedding": true
}

In [9]:
# Load bulk RNA-seq data and preprocess them.
csv_path = hf_hub_download(
    repo_id="InstaDeepAI/BulkRNABert",
    filename="data/tcga_sample.csv",
    repo_type="model",
)
gene_expression_array = pd.read_csv(csv_path).drop(["identifier"], axis=1).to_numpy()[:1, :]
gene_expression_array = np.log10(1 + gene_expression_array)
assert gene_expression_array.shape[1] == config.n_genes

# Tokenize
gene_expression_ids = tokenizer.batch_encode_plus(gene_expression_array, return_tensors="pt")["input_ids"]



NameError: name 'config' is not defined

In [ ]:
# Compute BulkRNABert's embeddings
gene_expression_mean_embeddings = model(gene_expression_ids)["embeddings_4"].mean(axis=1)  # embeddings can be used for downstream tasks.